# BWF Player Lookup

Type a badminton player's name and get their **personal details** (nationality, height, playing hand) and **ranking** (current rank and how long they have held it) from bwfbadminton.com.

**How to use:** set `PLAYER_NAME` in section 1, then *Run All*. Every value the site does not list is shown as `null`, with a note.

The notebook is a thin interface: all logic lives in the `bwf_player` package (see the README).

In [1]:
import logging

from bwf_player import BlockedByCloudflareError, BwfClientError, format_result, lookup_player

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## 1. Enter a player name

In [2]:
# Any reasonable spelling works: case, accents, reversed order, small typos ("jonathan cristie").
PLAYER_NAME = "jonathan cristie"

# None = the first ranking event the site lists (usually singles). To pick another event, copy the
# id shown as "Other event" in the result, e.g. "9-90070" (a doubles event).
EVENT_ID = None

## 2. Result

In [3]:
try:
    result = lookup_player(PLAYER_NAME, event_id=EVENT_ID)
except BlockedByCloudflareError as exc:
    result = None
    print(f"Blocked by Cloudflare. Wait a while (or switch network) before trying again.\n{exc}")
except BwfClientError as exc:
    result = None
    print(f"The request failed: {exc}")
else:
    print(format_result(result))

Search:         FOUND - Matched 'Jonatan CHRISTIE' (score 94).
Profile URL:    https://bwfbadminton.com/player/73442/jonatan-christie

Personal details
  Name:          Jonatan CHRISTIE
  Nationality:   Indonesia
  Height:        179.0 cm
  Playing hand:  Right

Ranking (MEN'S SINGLES)
  Current rank:  1
  At this rank:  4 week(s), since 2026-08-25 (latest ranking list 2026-09-15)


## 3. The same result as structured data (JSON)

In [4]:
print(result.model_dump_json(indent=2) if result else "No result.")

{
  "search": {
    "query": "jonathan cristie",
    "status": "found",
    "best_match": {
      "player_id": "73442",
      "slug": "jonatan-christie",
      "name": "Jonatan CHRISTIE",
      "country": null,
      "profile_url": "https://bwfbadminton.com/player/73442/jonatan-christie",
      "score": 93.8
    },
    "candidates": [
      {
        "player_id": "73442",
        "slug": "jonatan-christie",
        "name": "Jonatan CHRISTIE",
        "country": null,
        "profile_url": "https://bwfbadminton.com/player/73442/jonatan-christie",
        "score": 93.8
      }
    ],
    "message": "Matched 'Jonatan CHRISTIE' (score 94)."
  },
  "profile": {
    "player_id": "73442",
    "player_found": true,
    "name": "Jonatan CHRISTIE",
    "nationality": "Indonesia",
    "height_cm": 179.0,
    "playing_hand": "Right",
    "missing_fields": [],
    "notes": []
  },
  "ranking": {
    "player_id": "73442",
    "event": {
      "id": "6-0",
      "name": "MEN'S SINGLES"
    },
    "o

## More examples

The same lookup for other cases: a left-handed player, a retired (unranked) player, a player whose profile lists nothing, a name that matches several players, and a name that matches nobody.

In [5]:
EXAMPLES = ["Carolina Marin", "Chong Wei Lee", "Aadhya Shine", "christie", "not a real player"]

for name in EXAMPLES:
    print("=" * 72)
    print(f"Query: {name!r}")
    try:
        print(format_result(lookup_player(name)))
    except BwfClientError as exc:
        print(f"The request failed: {exc}")

INFO bwf_player.http_client: Bootstrapping session cookie


Query: 'Carolina Marin'
Search:         FOUND - Matched 'Carolina MARIN' (score 100).
Profile URL:    https://bwfbadminton.com/player/18228/carolina-marin

Personal details
  Name:          Carolina MARIN
  Nationality:   Spain
  Height:        172.0 cm
  Playing hand:  Left

Ranking (WOMEN'S SINGLES)
  Current rank:  null
  At this rank:  null
  Other event:   WOMEN'S DOUBLES (Beatriz CORRALES) [9-95780]
  Other event:   WOMEN'S DOUBLES (Sara PEÑALVER) [9-76747]
  Other event:   WOMEN'S DOUBLES (Isabel FERNANDEZ) [9-90558]
  Other event:   WOMEN'S DOUBLES (Clara AZURMENDI) [9-74218]
  Other event:   WOMEN'S DOUBLES (Ana Maria MARTIN) [9-55582]

Notes
  - Not currently ranked in WOMEN'S SINGLES: the site lists no current rank.
Query: 'Chong Wei Lee'
Search:         FOUND - Matched 'LEE Chong Wei' (score 100).
Profile URL:    https://bwfbadminton.com/player/50152/lee-chong-wei

Personal details
  Name:          LEE Chong Wei
  Nationality:   Malaysia
  Height:        172.0 cm
  Playing 

Search:         NOT_FOUND - No player matched 'not a real player' at or above 85. Closest was 'Emil DANTLER' (62).


## Notes

- **Search** ignores case, accents and word order, and tolerates typos in full names. A partial name such as "christie" returns ranked candidates instead of guessing.
- **Weeks at this rank** is derived from the site's weekly ranking history: the number of consecutive weekly lists, ending with the latest, that show the current rank. (The site's own "consecutive weeks" figure describes the player's *best* rank, so it is not used.)
- **null** means the site lists no value. A player with no current rank (retired, or inactive for a long time) is reported as unranked.
- Responses are cached under `.cache/bwf_player` (ranking and profile for 24 h, the player index for 7 days), so re-running a cell makes no new requests. The site sits behind Cloudflare bot protection; the client rate-limits itself and stops at once if it is blocked.
- Known limitations and the full design are in `docs/PRD_master.md`.